# Muscle Bulk RNA-seq Analysis — Harmonised Pipeline (v2)
## STAR + Salmon / GRCm39 — matching kidney pipeline exactly

This notebook re-processes muscle bulk RNA-seq from raw FASTQs using:
- TrimGalore → STAR (v2.7.10a) → Salmon quantification from transcriptome BAM
- GRCm39 / Ensembl 110 reference (same as kidney)
- tximport → DESeq2 with lfcShrink/ashr

This replaces the original FeatureCounts/GRCm38 analysis to eliminate pipeline asymmetry.

## 1. Load libraries

In [ ]:
library(Matrix)
library(DESeq2)
library(tximport)
library(GenomicFeatures)
library(org.Mm.eg.db)
library(AnnotationDbi)
library(ggplot2)
library(pheatmap)
library(dplyr)
library(tidyr)
library(ashr)
library(clusterProfiler)
library(fgsea)
library(msigdbr)
library(reticulate)
library(Seurat)

## 2. Import Salmon quantifications via tximport
### (Matching kidney pipeline: STAR alignment → Salmon quant from transcriptome BAM → tximport)

In [ ]:
# ── Paths ────────────────────────────────────────────────────────
base_dir <- "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/v2_manual/salmon_out"
out_dir  <- "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# ── Sample table ────────────────────────────────────────────────
samples <- c("ctrl_1","ctrl_2","ctrl_3",
             "age_1","age_2","age_3",
             "DR_1","DR_2","DR_3",
             "sAct_1","sAct_2","sAct_3",
             "combi_1","combi_2","combi_3")

conditions <- rep(c("ctrl","age","DR","sAct","combi"), each = 3)

files <- file.path(base_dir, samples, "quant.sf")
names(files) <- samples

# Verify all files exist
cat("Files found:", sum(file.exists(files)), "of", length(files), "\n")
stopifnot(all(file.exists(files)))

In [ ]:
# ── Build tx2gene from GTF (same GTF as kidney) ─────────────────
gtf_path <- "/data/bonn-epyc/projects/manuela/genomes/Mus_musculus.GRCm39.110.gtf.gz"
if (!file.exists(gtf_path)) {
    gtf_path <- sub(".gz$", "", gtf_path)
}
cat("Using GTF:", gtf_path, "\n")

txdb <- makeTxDbFromGFF(gtf_path)
k <- keys(txdb, keytype = "TXNAME")
tx2gene <- AnnotationDbi::select(txdb, k, "GENEID", "TXNAME")

cat("Transcripts in tx2gene:", nrow(tx2gene), "\n")
cat("Unique genes:", length(unique(tx2gene$GENEID)), "\n")

In [ ]:
# ── Import via tximport ──────────────────────────────────────────
txi <- tximport(files, type = "salmon", tx2gene = tx2gene,
                ignoreTxVersion = TRUE, ignoreAfterBar = TRUE)

cat("Genes imported:", nrow(txi$counts), "\n")
cat("Samples:", ncol(txi$counts), "\n")

## 3. Create DESeq2 object

In [ ]:
# ── DESeq2 object from tximport ──────────────────────────────────
sample_info <- data.frame(
    condition = factor(conditions, levels = c("ctrl","age","DR","sAct","combi")),
    row.names = samples
)

dds <- DESeqDataSetFromTximport(txi, colData = sample_info, design = ~condition)

# Remove NA rows
dds <- dds[!is.na(rownames(dds)), ]

# Minimum expression filter (same as kidney: >=10 total counts)
keep <- rowSums(counts(dds)) >= 10
dds <- dds[keep, ]
cat("Genes after filtering:", nrow(dds), "\n")

# Run DESeq2
dds <- DESeq(dds)

## 4. QC: PCA and sample distances

In [ ]:
# ── Variance stabilising transformation ─────────────────────────
vsd <- vst(dds, blind = TRUE)

# PCA
plotPCA(vsd, intgroup = "condition") +
    ggtitle("PCA \u2014 Muscle bulk RNA-seq (STAR+Salmon/GRCm39)") +
    theme_minimal()

# Sample distance heatmap
sampleDists <- dist(t(assay(vsd)))
sampleDistMatrix <- as.matrix(sampleDists)
pheatmap(sampleDistMatrix,
         clustering_distance_rows = sampleDists,
         clustering_distance_cols = sampleDists,
         main = "Sample-to-sample distances")

## 5. Extract contrasts with lfcShrink (ashr)
### Matching kidney Cell 17: results() then lfcShrink(res=...)

In [ ]:
# ── Raw results first, then shrink (matching kidney exactly) ────
res_age_raw  <- results(dds, contrast = c("condition", "age", "ctrl"))
res_intv_raw <- results(dds, contrast = c("condition", "combi", "age"))
res_intv1_raw <- results(dds, contrast = c("condition", "DR", "age"))
res_intv2_raw <- results(dds, contrast = c("condition", "sAct", "age"))
res_combi_ctrl_raw <- results(dds, contrast = c("condition", "combi", "ctrl"))

# Shrunken LFCs
res_age  <- lfcShrink(dds, contrast = c("condition","age","ctrl"),
                       res = res_age_raw, type = "ashr")
res_intv <- lfcShrink(dds, contrast = c("condition","combi","age"),
                       res = res_intv_raw, type = "ashr")
res_intv1 <- lfcShrink(dds, contrast = c("condition","DR","age"),
                        res = res_intv1_raw, type = "ashr")
res_intv2 <- lfcShrink(dds, contrast = c("condition","sAct","age"),
                        res = res_intv2_raw, type = "ashr")
res_combi_ctrl <- lfcShrink(dds, contrast = c("condition","combi","ctrl"),
                             res = res_combi_ctrl_raw, type = "ashr")

cat("All contrasts computed.\n")
cat("Contrast summary:\n")
cat("  res_age:        age vs ctrl\n")
cat("  res_intv:       combi vs age\n")
cat("  res_intv1:      DR vs age\n")
cat("  res_intv2:      sAct vs age\n")
cat("  res_combi_ctrl: combi vs ctrl\n")

## 6. Rescue framework (Stage 0-2)
### Matching kidney Cells 18-19

In [ ]:
# ── Clean rownames and add symbols ────────────────────────────────
res_age_df <- as.data.frame(res_age)
rownames(res_age_df) <- sub("\\..*", "", rownames(res_age_df))
res_age_df$Symbol <- mapIds(org.Mm.eg.db, keys = rownames(res_age_df),
                             column = "SYMBOL", keytype = "ENSEMBL", multiVals = "first")

# Stage 0: Aging DEGs (matching kidney: padj < 0.05 & abs(log2FoldChange) > 0)
deg_age <- subset(res_age_df, padj < 0.05 & abs(log2FoldChange) > 0)

# Stage 1: Rescued \u2014 uses res_intv (combi vs age) DIRECTLY like kidney
rescued_genes <- deg_age[which(sign(deg_age$log2FoldChange) !=
    sign(res_intv[rownames(deg_age), 'log2FoldChange'])), ]

# Add combi vs ctrl for Stage 2
rescued_genes$padj_combi_ctrl <- res_combi_ctrl[rownames(rescued_genes), 'padj']
rescued_genes$log2FC_combi_ctrl <- res_combi_ctrl[rownames(rescued_genes), 'log2FoldChange']

# Stage 2: Rescued-to-normal
rescued_to_normal <- rescued_genes[abs(rescued_genes$log2FC_combi_ctrl) < 0.5, ]

cat("\n=== RESCUE FRAMEWORK ===\n")
cat("Aging DEGs:", nrow(deg_age), "\n")
cat("Rescued (Stage 1):", nrow(rescued_genes),
    "(", round(nrow(rescued_genes)/nrow(deg_age)*100, 1), "% of aging DEGs)\n")
cat("Not reversed:", nrow(deg_age) - nrow(rescued_genes), "\n")
cat("Rescued-to-normal (Stage 2):", nrow(rescued_to_normal),
    "(", round(nrow(rescued_to_normal)/nrow(rescued_genes)*100, 1), "% of rescued)\n")

## 7. Non-rescued genes
### Matching kidney Cell 19

In [ ]:
# ── Non-rescued: uses res_intv1/res_intv2 (vs age) DIRECTLY ─────
rescued_combi <- rownames(deg_age)[sign(deg_age$log2FoldChange) !=
    sign(res_intv[rownames(deg_age), 'log2FoldChange'])]
rescued_DR <- rownames(deg_age)[sign(deg_age$log2FoldChange) !=
    sign(res_intv1[rownames(deg_age), 'log2FoldChange'])]
rescued_sAct <- rownames(deg_age)[sign(deg_age$log2FoldChange) !=
    sign(res_intv2[rownames(deg_age), 'log2FoldChange'])]

rescued_all <- unique(c(rescued_combi, rescued_DR, rescued_sAct))
non_rescued_genes <- setdiff(rownames(deg_age), rescued_all)
non_rescued_df <- deg_age[non_rescued_genes, ]

# ── Add symbols to all dataframes ───────────────────────────────
add_gene_symbols <- function(df) {
    df$Symbol <- mapIds(org.Mm.eg.db, keys = rownames(df),
                         column = 'SYMBOL', keytype = 'ENSEMBL', multiVals = 'first')
    return(df)
}
rescued_genes <- add_gene_symbols(rescued_genes)
rescued_to_normal <- add_gene_symbols(rescued_to_normal)
deg_age <- add_gene_symbols(deg_age)
non_rescued_df <- add_gene_symbols(non_rescued_df)

cat("Non-rescued:", nrow(non_rescued_df),
    "(", round(nrow(non_rescued_df)/nrow(deg_age)*100, 1), "% of aging DEGs)\n")

## 8. Effect comparison and driver classification (Stage 3)
### Matching kidney Cells 26-27-28-30
### CRITICAL: LFC_DR and LFC_sAct are intervention vs AGE (not vs ctrl)

In [ ]:
# ── Effect comparison table ───────────────────────────────────────
reversal_genes <- rownames(rescued_to_normal)

effect_comparison <- data.frame(
    Gene = reversal_genes,
    LFC_Age = res_age[reversal_genes, 'log2FoldChange'],       # age vs ctrl
    LFC_Combined = res_intv[reversal_genes, 'log2FoldChange'],  # combi vs AGE
    LFC_DR = res_intv1[reversal_genes, 'log2FoldChange'],       # DR vs AGE
    LFC_sAct = res_intv2[reversal_genes, 'log2FoldChange']      # sAct vs AGE
)

# Add symbols
symbol_lookup <- rescued_to_normal$Symbol
names(symbol_lookup) <- rownames(rescued_to_normal)
effect_comparison$Symbol <- symbol_lookup[effect_comparison$Gene]
effect_comparison <- effect_comparison[, c("Gene", "Symbol",
    setdiff(colnames(effect_comparison), c("Gene", "Symbol")))]

# ── Restoration scores (matching kidney Cell 27) ────────────────
effect_comparison$Restoration_DR <- ifelse(
    sign(effect_comparison$LFC_DR) != sign(effect_comparison$LFC_Age),
    abs(effect_comparison$LFC_DR / effect_comparison$LFC_Age), 0)

effect_comparison$Restoration_sAct <- ifelse(
    sign(effect_comparison$LFC_sAct) != sign(effect_comparison$LFC_Age),
    abs(effect_comparison$LFC_sAct / effect_comparison$LFC_Age), 0)

effect_comparison$Restoration_Combined <- ifelse(
    sign(effect_comparison$LFC_Combined) != sign(effect_comparison$LFC_Age),
    abs(effect_comparison$LFC_Combined / effect_comparison$LFC_Age), 0)

# Quantify restoration strength
summary_stats <- data.frame(
    Mean_Restoration_DR = mean(effect_comparison$Restoration_DR, na.rm = TRUE),
    Mean_Restoration_sAct = mean(effect_comparison$Restoration_sAct, na.rm = TRUE),
    Mean_Restoration_Combined = mean(effect_comparison$Restoration_Combined, na.rm = TRUE)
)
print(summary_stats)

# ── Effect classification (matching kidney Cell 28) ─────────────
effect_comparison$Intervention_DR_Effect <- ifelse(
    sign(effect_comparison$LFC_DR) != sign(effect_comparison$LFC_Age), "Rescue", "Worsening")
effect_comparison$Intervention_sAct_Effect <- ifelse(
    sign(effect_comparison$LFC_sAct) != sign(effect_comparison$LFC_Age), "Rescue", "Worsening")
effect_comparison$Intervention_Combined_Effect <- ifelse(
    sign(effect_comparison$LFC_Combined) != sign(effect_comparison$LFC_Age), "Rescue", "Worsening")

# ── Driver categories (matching kidney Cell 30) ─────────────────
exclusive_DR <- effect_comparison %>% filter(Restoration_DR > 0 & Restoration_sAct == 0)
exclusive_sAct <- effect_comparison %>% filter(Restoration_sAct > 0 & Restoration_DR == 0)
only_rescued_in_combi <- effect_comparison %>% filter(Restoration_sAct == 0 & Restoration_DR == 0)

cat("\n=== DRIVER CLASSIFICATION ===\n")
cat("DR-exclusive:", nrow(exclusive_DR),
    "(", round(nrow(exclusive_DR)/nrow(rescued_to_normal)*100, 1), "%)\n")
cat("sAct-exclusive:", nrow(exclusive_sAct),
    "(", round(nrow(exclusive_sAct)/nrow(rescued_to_normal)*100, 1), "%)\n")
cat("Combi-only:", nrow(only_rescued_in_combi),
    "(", round(nrow(only_rescued_in_combi)/nrow(rescued_to_normal)*100, 1), "%)\n")
cat("Both:", nrow(rescued_to_normal) - nrow(exclusive_DR) - nrow(exclusive_sAct) - nrow(only_rescued_in_combi),
    "(", round((nrow(rescued_to_normal) - nrow(exclusive_DR) - nrow(exclusive_sAct) - nrow(only_rescued_in_combi))/nrow(rescued_to_normal)*100, 1), "%)\n")

## 9. Category proportions

In [ ]:
category_counts <- data.frame(
    Category = c("Exclusive_DR", "Exclusive_sAct", "Only_Active_in_Combi", "Other"),
    Count = c(nrow(exclusive_DR), nrow(exclusive_sAct),
              nrow(only_rescued_in_combi),
              nrow(rescued_to_normal) - nrow(exclusive_DR) - nrow(exclusive_sAct) - nrow(only_rescued_in_combi))
)
category_counts$Proportion <- category_counts$Count / sum(category_counts$Count)
print(category_counts)

## 10. Pipeline comparison

In [ ]:
n_both <- nrow(rescued_to_normal) - nrow(exclusive_DR) - nrow(exclusive_sAct) - nrow(only_rescued_in_combi)

cat("\n\u2554\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2557\n")
cat("\u2551     PIPELINE COMPARISON: MUSCLE BULK RNA-seq       \u2551\n")
cat("\u2560\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2563\n")
cat(sprintf("\u2551 %-28s %-13s %-13s \u2551\n", "", "FeatureCounts", "STAR+Salmon"))
cat(sprintf("\u2551 %-28s %-13s %-13s \u2551\n", "", "GRCm38", "GRCm39"))
cat("\u2560\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2563\n")
cat(sprintf("\u2551 %-28s %-13d %-13d \u2551\n", "Genes tested", 21055, nrow(res_age_df)))
cat(sprintf("\u2551 %-28s %-13d %-13d \u2551\n", "Aging DEGs", 1739, nrow(deg_age)))
cat(sprintf("\u2551 %-28s %-13d %-13d \u2551\n", "  Up", 881, sum(deg_age$log2FoldChange > 0)))
cat(sprintf("\u2551 %-28s %-13d %-13d \u2551\n", "  Down", 858, sum(deg_age$log2FoldChange < 0)))
cat("\u2560\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2563\n")
cat(sprintf("\u2551 %-28s %-13d %-13d \u2551\n", "Rescued (Stage 1)", 926, nrow(rescued_genes)))
cat(sprintf("\u2551 %-28s %-13d %-13d \u2551\n", "  Not reversed", 813, nrow(deg_age) - nrow(rescued_genes)))
cat(sprintf("\u2551 %-28s %-13d %-13d \u2551\n", "Rescued-to-normal (Stage 2)", 511, nrow(rescued_to_normal)))
cat(sprintf("\u2551 %-28s %-13d %-13d \u2551\n", "  DR-exclusive", 241, nrow(exclusive_DR)))
cat(sprintf("\u2551 %-28s %-13d %-13d \u2551\n", "  sAct-exclusive", 23, nrow(exclusive_sAct)))
cat(sprintf("\u2551 %-28s %-13d %-13d \u2551\n", "  Combi-only", 18, nrow(only_rescued_in_combi)))
cat(sprintf("\u2551 %-28s %-13d %-13d \u2551\n", "  Both DR + sAct", 229, n_both))
cat(sprintf("\u2551 %-28s %-13d %-13d \u2551\n", "Non-rescued", 300, nrow(non_rescued_df)))
cat("\u255a\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u255d\n")

## 11. Gene overlap with original pipeline

In [ ]:
# ── Aging DEG overlap ────────────────────────────────────────────
orig <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/1st_attempt_count_table_data/results_tables/muscle_res_age_results_with_symbols.csv")
orig_sig <- orig$Symbol[!is.na(orig$padj) & orig$padj < 0.05 & !is.na(orig$Symbol)]
new_sig <- res_age_df$Symbol[!is.na(res_age_df$padj) & res_age_df$padj < 0.05 & !is.na(res_age_df$Symbol)]
shared <- intersect(orig_sig, new_sig)

cat(sprintf("\nAging DEG overlap (STAR+Salmon vs Original):\n"))
cat(sprintf("  Original only:  %d\n", length(setdiff(orig_sig, new_sig))))
cat(sprintf("  New only:       %d\n", length(setdiff(new_sig, orig_sig))))
cat(sprintf("  Shared:         %d (%.0f%% of original, %.0f%% of new)\n",
    length(shared), length(shared)/length(orig_sig)*100, length(shared)/length(new_sig)*100))

# ── Rescued-to-normal overlap ───────────────────────────────────
orig_rescued <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/1st_attempt_count_table_data/results_tables/muscle_rescued_to_normal.csv")
if ("Symbol" %in% colnames(orig_rescued)) {
    orig_rtn <- orig_rescued$Symbol[!is.na(orig_rescued$Symbol)]
    new_rtn <- rescued_to_normal$Symbol[!is.na(rescued_to_normal$Symbol)]
    shared_rtn <- intersect(orig_rtn, new_rtn)
    cat(sprintf("\nRescued-to-normal overlap:\n"))
    cat(sprintf("  Original only:  %d\n", length(setdiff(orig_rtn, new_rtn))))
    cat(sprintf("  New only:       %d\n", length(setdiff(new_rtn, orig_rtn))))
    cat(sprintf("  Shared:         %d (%.0f%% of original, %.0f%% of new)\n",
        length(shared_rtn), length(shared_rtn)/length(orig_rtn)*100, length(shared_rtn)/length(new_rtn)*100))
}

## 12. GO Biological Process enrichment
### Matching kidney Cell 43

In [ ]:
all_genes <- rownames(res_age_df)

# Aging
ego_age <- enrichGO(gene = rownames(deg_age), universe = all_genes,
                     OrgDb = org.Mm.eg.db, keyType = "ENSEMBL", ont = "BP",
                     pAdjustMethod = "BH", pvalueCutoff = 0.05, qvalueCutoff = 0.2)

# Rescued-to-normal
ego_rescued <- enrichGO(gene = rownames(rescued_to_normal), universe = all_genes,
                          OrgDb = org.Mm.eg.db, keyType = "ENSEMBL", ont = "BP",
                          pAdjustMethod = "BH", pvalueCutoff = 0.05, qvalueCutoff = 0.2)

# Non-rescued
ego_non_rescued <- enrichGO(gene = rownames(non_rescued_df), universe = all_genes,
                              OrgDb = org.Mm.eg.db, keyType = "ENSEMBL", ont = "BP",
                              pAdjustMethod = "BH", pvalueCutoff = 0.05, qvalueCutoff = 0.2)

# DR-exclusive
ego_DR <- enrichGO(gene = exclusive_DR$Gene, OrgDb = org.Mm.eg.db,
                    keyType = "ENSEMBL", ont = "BP",
                    pAdjustMethod = "BH", pvalueCutoff = 0.05, qvalueCutoff = 0.2)

# sAct-exclusive (relaxed cutoff)
ego_sAct <- enrichGO(gene = exclusive_sAct$Gene, OrgDb = org.Mm.eg.db,
                      keyType = "ENSEMBL", ont = "BP",
                      pAdjustMethod = "BH", pvalueCutoff = 0.2, qvalueCutoff = 0.2)

# Combi-only
ego_combi_only <- enrichGO(gene = only_rescued_in_combi$Gene, OrgDb = org.Mm.eg.db,
                             keyType = "ENSEMBL", ont = "BP",
                             pAdjustMethod = "BH", pvalueCutoff = 0.2, qvalueCutoff = 0.2)

cat("\n=== GOBP ENRICHMENT ===\n")
cat("Aging terms:", nrow(as.data.frame(ego_age)), "\n")
cat("Rescued terms:", nrow(as.data.frame(ego_rescued)), "\n")
cat("Non-rescued terms:", nrow(as.data.frame(ego_non_rescued)), "\n")
cat("DR-exclusive terms:", nrow(as.data.frame(ego_DR)), "\n")
cat("sAct-exclusive terms:", nrow(as.data.frame(ego_sAct)), "\n")
cat("Combi-only terms:", nrow(as.data.frame(ego_combi_only)), "\n")

cat("\nTop 5 aging GOBP:\n")
print(head(as.data.frame(ego_age)[, c("Description","Count","p.adjust")], 5))

## 13. GSEA
### Matching kidney Cell 46

In [ ]:
# ── Convert DESeq2 results to clean data.frames ─────────────────────────────
# ── Convert to dataframes ───────────────────────────────────────

res_age_gsea <- as.data.frame(res_age)
res_age_gsea$ENSEMBL <- rownames(res_age_gsea)
res_age_gsea$ENSEMBL_clean <- sub("\\..*$", "", res_age_gsea$ENSEMBL)

res_age_gsea$Symbol <- mapIds(
    org.Mm.eg.db,
    keys = res_age_gsea$ENSEMBL_clean,
    column = "SYMBOL",
    keytype = "ENSEMBL",
    multiVals = "first"
)

res_intv_gsea <- as.data.frame(res_intv)
res_intv_gsea$ENSEMBL <- rownames(res_intv_gsea)
res_intv_gsea$ENSEMBL_clean <- sub("\\..*$", "", res_intv_gsea$ENSEMBL)

res_intv_gsea$Symbol <- mapIds(
    org.Mm.eg.db,
    keys = res_intv_gsea$ENSEMBL_clean,
    column = "SYMBOL",
    keytype = "ENSEMBL",
    multiVals = "first"
)

# ── IMPORTANT FIX:
# use log2FoldChange instead of stat ─────────────────────────────

ranked_age <- res_age_gsea[
    !is.na(res_age_gsea$log2FoldChange) &
    !is.na(res_age_gsea$Symbol) &
    res_age_gsea$Symbol != "",
]

ranked_age <- ranked_age[
    order(ranked_age$log2FoldChange, decreasing = TRUE),
]

ranks_age <- setNames(
    ranked_age$log2FoldChange,
    ranked_age$Symbol
)

ranks_age <- ranks_age[!duplicated(names(ranks_age))]
ranks_age <- ranks_age[is.finite(ranks_age)]

ranked_intv <- res_intv_gsea[
    !is.na(res_intv_gsea$log2FoldChange) &
    !is.na(res_intv_gsea$Symbol) &
    res_intv_gsea$Symbol != "",
]

ranked_intv <- ranked_intv[
    order(ranked_intv$log2FoldChange, decreasing = TRUE),
]

ranks_intv <- setNames(
    ranked_intv$log2FoldChange,
    ranked_intv$Symbol
)

ranks_intv <- ranks_intv[!duplicated(names(ranks_intv))]
ranks_intv <- ranks_intv[is.finite(ranks_intv)]

# ── MSigDB GO Biological Process pathways ───────────────────────────────────

m_df <- msigdbr(
    species = "Mus musculus",
    category = "C5",
    subcategory = "GO:BP"
)

pathways <- split(
    m_df$gene_symbol,
    m_df$gs_name
)

cat("Loaded", length(pathways), "GO:BP pathways\n")

# ── Run fgsea ───────────────────────────────────────────────────────────────

fg_age <- fgsea(
    pathways = pathways,
    stats = ranks_age,
    minSize = 15,
    maxSize = 500
)

fg_intv <- fgsea(
    pathways = pathways,
    stats = ranks_intv,
    minSize = 15,
    maxSize = 500
)

# ── Summary ─────────────────────────────────────────────────────────────────

cat("\n=== GSEA significant pathways ===\n")

cat(
    "Aging:",
    sum(
        fg_age$padj < 0.05 &
        abs(fg_age$NES) > 1.5,
        na.rm = TRUE
    ),
    "\n"
)

cat(
    "Combi:",
    sum(
        fg_intv$padj < 0.05 &
        abs(fg_intv$NES) > 1.5,
        na.rm = TRUE
    ),
    "\n"
)

# ── NES comparison matrix ───────────────────────────────────────────────────

fg_age_sig <- fg_age %>%
    filter(!is.na(padj)) %>%
    filter(padj < 0.05) %>%
    arrange(padj)

top_pathways <- fg_age_sig %>%
    head(30) %>%
    pull(pathway)

fg_age_unique <- fg_age %>%
    filter(pathway %in% top_pathways) %>%
    distinct(pathway, .keep_all = TRUE) %>%
    dplyr::select(
        pathway,
        NES_age = NES
    )

fg_intv_unique <- fg_intv %>%
    filter(pathway %in% top_pathways) %>%
    distinct(pathway, .keep_all = TRUE) %>%
    dplyr::select(
        pathway,
        NES_intv = NES
    )

nes_mat <- fg_age_unique %>%
    left_join(
        fg_intv_unique,
        by = "pathway"
    )

cat("\n=== Top pathway NES comparison ===\n")
print(head(nes_mat, 10))

## 14. Visualisation

In [ ]:
# ── MA plot ──────────────────────────────────────────────────────
plotMA(res_age, main = "MA Plot: Age vs Ctrl (Muscle, STAR+Salmon/GRCm39)", ylim = c(-5, 5))

# ── Normalised counts for top gene ──────────────────────────────
ntd <- normTransform(dds)
top_gene <- rownames(deg_age)[which.min(deg_age$padj)]
plotCounts(dds, gene = top_gene, intgroup = "condition",
           main = paste("Top aging DEG:", deg_age[top_gene, "Symbol"]))

In [ ]:
# ── Volcano plot ─────────────────────────────────────────────────
volcano_df <- res_age_df
volcano_df$neg_log10_padj <- -log10(pmax(volcano_df$padj, 1e-310))
volcano_df$color <- ifelse(is.na(volcano_df$padj) | volcano_df$padj >= 0.05, "grey70",
                    ifelse(volcano_df$log2FoldChange > 0, "#A32D2D", "#185FA5"))

ggplot(volcano_df, aes(x = log2FoldChange, y = neg_log10_padj, color = color)) +
    geom_point(size = 0.5, alpha = 0.4) +
    scale_color_identity() +
    geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey50") +
    geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
    labs(title = "Muscle v2 \u2014 Aging DEGs (STAR+Salmon/GRCm39)",
         x = expression(log[2]~fold~change), y = expression(-log[10]~p[adj])) +
    theme_minimal() +
    annotate("text", x = 3, y = max(volcano_df$neg_log10_padj, na.rm = TRUE) * 0.9,
             label = sprintf("%d up", sum(deg_age$log2FoldChange > 0)), color = "#A32D2D") +
    annotate("text", x = -3, y = max(volcano_df$neg_log10_padj, na.rm = TRUE) * 0.9,
             label = sprintf("%d down", sum(deg_age$log2FoldChange < 0)), color = "#185FA5")

## 15. Export all results

In [ ]:
# ── DEG results ──────────────────────────────────────────────────
write.csv(res_age_df, file.path(out_dir, "muscle_v2_res_age_results_with_symbols.csv"))
write.csv(as.data.frame(rescued_to_normal), file.path(out_dir, "muscle_v2_rescued_to_normal.csv"))
write.csv(as.data.frame(non_rescued_df), file.path(out_dir, "muscle_v2_non_rescued_aged_DEGs.csv"))
write.csv(as.data.frame(deg_age), file.path(out_dir, "muscle_v2_deg_age.csv"))
write.csv(as.data.frame(rescued_genes), file.path(out_dir, "muscle_v2_combined_rescued_genes_DE_info.csv"))

# ── Effect comparison + categories ───────────────────────────────
write.csv(effect_comparison, file.path(out_dir, "muscle_v2_intervention_impact_comparison.csv"), row.names = FALSE)
write.csv(category_counts, file.path(out_dir, "muscle_v2_gene_category_proportions.csv"), row.names = FALSE)

# ── Driver-specific gene lists ──────────────────────────────────
write.csv(effect_comparison[effect_comparison$Gene %in% exclusive_DR$Gene, ],
          file.path(out_dir, "muscle_v2_intervention_impact_comparison_DR.csv"), row.names = FALSE)
write.csv(effect_comparison[effect_comparison$Gene %in% exclusive_sAct$Gene, ],
          file.path(out_dir, "muscle_v2_intervention_impact_comparison_sAct.csv"), row.names = FALSE)
write.csv(effect_comparison[effect_comparison$Gene %in% only_rescued_in_combi$Gene, ],
          file.path(out_dir, "muscle_v2_intervention_impact_comparison_only_rescued_in_combi.csv"), row.names = FALSE)

# ── Enrichment results ──────────────────────────────────────────
write.csv(as.data.frame(ego_age), file.path(out_dir, "muscle_v2_enrichment_results_aging.csv"), row.names = FALSE)
write.csv(as.data.frame(ego_rescued), file.path(out_dir, "muscle_v2_enrichment_results_rescued_to_normal.csv"), row.names = FALSE)
write.csv(as.data.frame(ego_non_rescued), file.path(out_dir, "muscle_v2_enrichment_results_non_rescued_genes.csv"), row.names = FALSE)
write.csv(as.data.frame(ego_DR), file.path(out_dir, "muscle_v2_enrichment_results_rescued-to-normal__DR.csv"), row.names = FALSE)
write.csv(as.data.frame(ego_sAct), file.path(out_dir, "muscle_v2_enrichment_results_rescued-to-normal_sAct.csv"), row.names = FALSE)
write.csv(as.data.frame(ego_combi_only), file.path(out_dir, "muscle_v2_enrichment_results_rescued-to-normal_only_in_combi.csv"), row.names = FALSE)

# ── GSEA results ─────────────────────────────────────────────────
fg_age_df <- fg_age
fg_age_df$leadingEdge <- sapply(fg_age_df$leadingEdge, paste, collapse = ";")

fg_intv_df <- fg_intv
fg_intv_df$leadingEdge <- sapply(fg_intv_df$leadingEdge, paste, collapse = ";")

write.csv(fg_age_df,
          file.path(out_dir, "muscle_v2_fgsea_aging.csv"),
          row.names = FALSE)
write.csv(as.data.frame(fg_intv_df), file.path(out_dir, "muscle_v2_fgsea_combi.csv"), row.names = FALSE)

cat("\nAll results exported to:", out_dir, "\n")
cat("Files:\n")
cat(paste(" ", list.files(out_dir), collapse = "\n"), "\n")

In [ ]:
# ── Export gene lists for DAVID web tool ─────────────────────────
# Upload to https://david.ncifcrf.gov → Start Analysis → Upload
# Select: Gene List, Identifier = OFFICIAL_GENE_SYMBOL, Species = Mus musculus

david_dir <- file.path(out_dir, "DAVID_input")
dir.create(david_dir, recursive = TRUE, showWarnings = FALSE)

# ── 1. Aging DEGs |LFC| > 1 ─────────────────────────────────────
aging_lfc1 <- deg_age[abs(deg_age$log2FoldChange) > 1 & !is.na(deg_age$Symbol), ]
writeLines(na.omit(aging_lfc1$Symbol),
           file.path(david_dir, "muscle_v2_aged_lfc1_symbols.txt"))
cat("Aging |LFC|>1 genes:", length(na.omit(aging_lfc1$Symbol)), "\n")

# ── 2. Rescued-to-normal |LFC| > 1 ──────────────────────────────
rescued_lfc1 <- rescued_to_normal[abs(rescued_to_normal$log2FoldChange) > 1 &
                                   !is.na(rescued_to_normal$Symbol), ]
writeLines(na.omit(rescued_lfc1$Symbol),
           file.path(david_dir, "muscle_v2_rescued_to_normal_lfc1_symbols.txt"))
cat("Rescued-to-normal |LFC|>1 genes:", length(na.omit(rescued_lfc1$Symbol)), "\n")

# ── 3. DR-exclusive |LFC| > 1 ───────────────────────────────────
dr_lfc1 <- exclusive_DR[abs(exclusive_DR$LFC_Age) > 1 & !is.na(exclusive_DR$Symbol), ]
writeLines(na.omit(dr_lfc1$Symbol),
           file.path(david_dir, "muscle_v2_DR_exclusive_lfc1_symbols.txt"))
cat("DR-exclusive |LFC|>1 genes:", length(na.omit(dr_lfc1$Symbol)), "\n")

# ── 4. Non-rescued |LFC| > 1 ────────────────────────────────────
nonrescued_lfc1 <- non_rescued_df[abs(non_rescued_df$log2FoldChange) > 1 &
                                   !is.na(non_rescued_df$Symbol), ]
writeLines(na.omit(nonrescued_lfc1$Symbol),
           file.path(david_dir, "muscle_v2_non_rescued_lfc1_symbols.txt"))
cat("Non-rescued |LFC|>1 genes:", length(na.omit(nonrescued_lfc1$Symbol)), "\n")

cat("\nAll DAVID input files saved to:", david_dir, "\n")
cat("Files:\n")
cat(paste(" ", list.files(david_dir)), sep="\n")

In [ ]:
# ── Per-cell-type GOBP enrichment for RESCUED genes (v2) ─────────
library(clusterProfiler)
library(org.Mm.eg.db)

# Load v2 reversal gene markers from FindAllMarkers
markers_reversal_v2 <- read.csv(
    "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon/muscle_v2_markers_reversal_genes_celltypes.csv"
)

# Universe = all genes tested in v2 DESeq2
res_age_v2 <- read.csv(
    "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon/muscle_v2_res_age_results_with_symbols.csv"
)
all_genes_v2 <- sub("\\.[0-9]+$", "", res_age_v2$X)  # clean Ensembl IDs

# Output directory — mirror original subdir structure
out_base <- "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon/GOBP_per_celltype/rescued"

# Create subdirs
for (sub in c("ECs", "IMM", "NMJs", "Rest")) {
    dir.create(file.path(out_base, sub), recursive = TRUE, showWarnings = FALSE)
}

# Map cell type → subdir (matching original structure)
ct_subdir <- list(
    "Endothelial cells"                          = "ECs",
    "Lymphatic vessels"                          = "ECs",
    "Macrophages"                                = "IMM",
    "Immune cells?"                              = "IMM",
    "Myonuclei (neuromuscular junction nuclei)"  = "NMJs",
    "Myonuclei (myotendinous junction nuclei)"   = "NMJs",
    "Myonuclei"                                  = "Rest",
    "Myonuclei (Myh1+)"                          = "Rest",
    "Myonuclei (Myh4+?)"                         = "Rest",
    "Myonuclei (differentiating myocytes?)"      = "Rest",
    "FAPs"                                       = "Rest",
    "Fibroblasts"                                = "Rest",
    "Muscle stem cell"                           = "Rest",
    "Tenocytes"                                  = "Rest"
)

sanitize <- function(x) gsub("[^A-Za-z0-9_.() +-]", "_", x)

cell_types <- unique(markers_reversal_v2$cluster)
cat("Cell types to process:", length(cell_types), "\n\n")

for (ct in cell_types) {
    cat("Processing:", ct, "\n")

    ct_markers <- markers_reversal_v2[markers_reversal_v2$cluster == ct, "gene"]

    if (length(ct_markers) < 5) {
        cat("  Skipping — fewer than 5 genes\n"); next
    }

    # Map symbols to Ensembl
    ct_ensembl <- mapIds(org.Mm.eg.db, keys = ct_markers,
                          column = "ENSEMBL", keytype = "SYMBOL", multiVals = "first")
    ct_ensembl <- na.omit(ct_ensembl)

    if (length(ct_ensembl) < 5) {
        cat("  Skipping — fewer than 5 mappable genes\n"); next
    }

    ego <- tryCatch(
        enrichGO(gene          = ct_ensembl,
                 universe      = all_genes_v2,
                 OrgDb         = org.Mm.eg.db,
                 keyType       = "ENSEMBL",
                 ont           = "BP",
                 pAdjustMethod = "BH",
                 pvalueCutoff  = 0.05,
                 qvalueCutoff  = 0.2),
        error = function(e) { cat("  Error:", conditionMessage(e), "\n"); NULL }
    )

    if (!is.null(ego) && nrow(as.data.frame(ego)) > 0) {
        subdir  <- ct_subdir[[ct]]
        if (is.null(subdir)) subdir <- "Rest"
        ct_safe <- sanitize(ct)
        out_path <- file.path(out_base, subdir,
                               paste0("gobp_rescued_", ct_safe, ".csv"))
        write.csv(as.data.frame(ego), out_path, row.names = FALSE)
        cat("  Saved:", nrow(as.data.frame(ego)), "terms →", out_path, "\n")
    } else {
        cat("  No significant terms\n")
    }
}

cat("\nDone. Files in output:\n")
for (sub in c("ECs", "IMM", "NMJs", "Rest")) {
    files <- list.files(file.path(out_base, sub))
    cat(sub, ":", length(files), "files\n")
    for (f in files) cat("  ", f, "\n")
}

# sn integration

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MUSCLE v2 — snRNA-seq integration (using v2 bulk gene lists)
# Same Seurat workflow as original, only gene lists differ
# ══════════════════════════════════════════════════════════════════

# ── 1. Load h5ad → Seurat (unchanged from original) ─────────────
use_condaenv("r-deseq2-env", required = TRUE)
anndata <- import("anndata")
adata <- anndata$read_h5ad("/data/bonn-epyc/projects/manuela/aging/aging_muscle/annotated-muscle-soupxed2.h5ad")

count_matrix <- t(as.matrix(adata$X))
gene_names <- py_to_r(adata$var_names$tolist())
rownames(count_matrix) <- gene_names
cell_metadata <- as.data.frame(py_to_r(adata$obs))
rownames(cell_metadata) <- colnames(count_matrix)

seurat_obj <- CreateSeuratObject(counts = count_matrix, meta.data = cell_metadata)

if (!is.null(adata$obsm)) {
    pca_mat <- py_to_r(adata$obsm[["X_pca"]])
    umap_mat <- py_to_r(adata$obsm[["X_umap"]])
    rownames(pca_mat) <- colnames(seurat_obj)
    rownames(umap_mat) <- colnames(seurat_obj)
    seurat_obj[["pca"]] <- CreateDimReducObject(embeddings = pca_mat, key = "PC_",
                                                 assay = DefaultAssay(seurat_obj))
    seurat_obj[["umap"]] <- CreateDimReducObject(embeddings = umap_mat, key = "UMAP_",
                                                  assay = DefaultAssay(seurat_obj))
}

# Normalize (same as original)
counts_mat <- LayerData(seurat_obj[["RNA"]], layer = "counts")
total_counts <- Matrix::colSums(counts_mat)
nonzero_cells <- total_counts > 0
norm_counts <- Matrix(0, nrow = nrow(counts_mat), ncol = ncol(counts_mat),
                       dimnames = dimnames(counts_mat))
norm_counts[, nonzero_cells] <- sweep(counts_mat[, nonzero_cells], 2,
                                       total_counts[nonzero_cells], "/") * 1e4
log_norm <- log1p(norm_counts)
log_norm[is.na(log_norm)] <- 0
log_norm[is.infinite(log_norm)] <- 0
LayerData(seurat_obj[["RNA"]], layer = "data") <- log_norm
DefaultLayer(seurat_obj[["RNA"]]) <- "data"

Idents(seurat_obj) <- seurat_obj$celltype
cat("Seurat object ready:", ncol(seurat_obj), "cells,",
    length(unique(seurat_obj$celltype)), "cell types\n")

# ── 2. Prepare v2 gene lists ────────────────────────────────────
# These come from the v2 STAR+Salmon DESeq2 analysis (already in memory)
reversal_genes_v2 <- na.omit(unique(rescued_to_normal$Symbol))
genes_aged_v2 <- na.omit(unique(deg_age$Symbol))
genes_DR_v2 <- na.omit(unique(exclusive_DR$Symbol))
genes_sAct_v2 <- na.omit(unique(exclusive_sAct$Symbol))

# Filter to genes present in Seurat object
features_all <- rownames(seurat_obj)
genes_norm_v2 <- unique(reversal_genes_v2[reversal_genes_v2 %in% features_all])
genes_aged_v2 <- unique(genes_aged_v2[genes_aged_v2 %in% features_all])
genes_DR_v2 <- unique(genes_DR_v2[genes_DR_v2 %in% features_all])
genes_sAct_v2 <- unique(genes_sAct_v2[genes_sAct_v2 %in% features_all])

cat("\nv2 gene lists in Seurat object:\n")
cat("  Reversal (rescued-to-normal):", length(genes_norm_v2), "\n")
cat("  Aged:", length(genes_aged_v2), "\n")
cat("  DR-exclusive:", length(genes_DR_v2), "\n")
cat("  sAct-exclusive:", length(genes_sAct_v2), "\n")

# ── Compare with original gene lists ────────────────────────────
# Load original rescued-to-normal for comparison
orig_rtn <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/1st_attempt_count_table_data/results_tables/muscle_rescued_to_normal.csv")
orig_rev <- na.omit(unique(orig_rtn$Symbol))
orig_rev_in_seurat <- orig_rev[orig_rev %in% features_all]

shared_rev <- intersect(genes_norm_v2, orig_rev_in_seurat)
cat("\nReversal gene overlap (in Seurat):\n")
cat("  Original:", length(orig_rev_in_seurat), "\n")
cat("  v2:", length(genes_norm_v2), "\n")
cat("  Shared:", length(shared_rev),
    sprintf("(%.0f%% of original, %.0f%% of v2)\n",
            length(shared_rev)/length(orig_rev_in_seurat)*100,
            length(shared_rev)/length(genes_norm_v2)*100))

# ── 3. AddModuleScore (same functions, v2 gene lists) ───────────
seurat_obj <- AddModuleScore(seurat_obj, features = list(genes_norm_v2), name = "Reversal_norm")
seurat_obj <- AddModuleScore(seurat_obj, features = list(genes_DR_v2), name = "Reversal_DR")
seurat_obj <- AddModuleScore(seurat_obj, features = list(genes_sAct_v2), name = "Reversal_sAct")
seurat_obj <- AddModuleScore(seurat_obj, features = list(genes_aged_v2), name = "Aged")

cat("\nModule scores added.\n")

# ── 4. Violin plots ─────────────────────────────────────────────
reorder_celltypes <- function(seurat_obj, feature) {
    avg_exp <- FetchData(seurat_obj, vars = c(feature, "celltype")) %>%
        group_by(celltype) %>%
        summarise(mean_exp = mean(get(feature), na.rm = TRUE)) %>%
        arrange(desc(mean_exp))
    seurat_obj$celltype <- factor(seurat_obj$celltype, levels = avg_exp$celltype)
    return(seurat_obj)
}

for (score_name in c("Reversal_norm1", "Reversal_DR1", "Reversal_sAct1", "Aged1")) {
    seurat_obj <- reorder_celltypes(seurat_obj, score_name)
    print(VlnPlot(seurat_obj, features = score_name, group.by = "celltype", pt.size = 0) +
              ggtitle(paste("v2 Module Score:", score_name)) +
              theme_classic() +
              theme(axis.text.x = element_text(angle = 90, hjust = 1, size = 12),
                    axis.title.x = element_blank(),
                    plot.title = element_text(size = 14),
                    axis.text.y = element_text(size = 12),
                    axis.title.y = element_text(size = 12)))
}

# ── 5. FindAllMarkers + overlap (matching kidney Cell 114) ───────
Idents(seurat_obj) <- seurat_obj$celltype
markers_all <- FindAllMarkers(seurat_obj, only.pos = TRUE, min.pct = 0.1, logfc.threshold = 0.25)

reversal_genes_clean <- reversal_genes_v2[!is.na(reversal_genes_v2) & reversal_genes_v2 != ""]
aged_genes_clean <- genes_aged_v2[!is.na(genes_aged_v2) & genes_aged_v2 != ""]
combi_only_clean <- na.omit(unique(only_rescued_in_combi$Symbol))
combi_only_clean <- combi_only_clean[combi_only_clean != ""]

markers_reversal <- markers_all[markers_all$gene %in% reversal_genes_clean, ]
markers_aged <- markers_all[markers_all$gene %in% aged_genes_clean, ]
markers_only_rescued_in_combi <- markers_all[markers_all$gene %in% combi_only_clean, ]

cat("\nMarker overlaps (v2):\n")
cat("  All markers:", nrow(markers_all), "\n")
cat("  Reversal gene markers:", nrow(markers_reversal), "\n")
cat("  Aged gene markers:", nrow(markers_aged), "\n")
cat("  Combi-only markers:", nrow(markers_only_rescued_in_combi), "\n")

# ── 6. Export v2 snRNA-seq results ───────────────────────────────
v2_out <- "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon"

# Module scores
df_scores <- data.frame(
    celltype      = seurat_obj$celltype,
    Reversal_norm = seurat_obj@meta.data$Reversal_norm1,
    Reversal_DR   = seurat_obj@meta.data$Reversal_DR1,
    Reversal_sAct = seurat_obj@meta.data$Reversal_sAct1,
    Aged          = seurat_obj@meta.data$Aged1
)
write.csv(df_scores, file.path(v2_out, "muscle_v2_module_scores_all.csv"), row.names = FALSE)

# Aging scores only
df_aging <- data.frame(
    celltype = seurat_obj$celltype,
    Aged1    = seurat_obj@meta.data$Aged1
)
write.csv(df_aging, file.path(v2_out, "muscle_v2_AgingGeneScores_perCell_seurat.csv"), row.names = FALSE)

# Markers
write.csv(markers_all, file.path(v2_out, "muscle_v2_markers_all_celltypes.csv"), row.names = FALSE)
write.csv(markers_reversal, file.path(v2_out, "muscle_v2_markers_reversal_genes_celltypes.csv"), row.names = FALSE)
write.csv(markers_aged, file.path(v2_out, "muscle_v2_markers_aged_genes_celltypes.csv"), row.names = FALSE)
write.csv(markers_only_rescued_in_combi, file.path(v2_out, "muscle_v2_markers_only_rescued_in_combi_celltypes.csv"), row.names = FALSE)

# Per-cell-type marker files
sanitize_name <- function(x) gsub("/", "_", x)
for (ct in sanitize_name(unique(markers_reversal$cluster))) {
    sub_df <- markers_reversal[sanitize_name(markers_reversal$cluster) == ct, ]
    write.csv(sub_df, file.path(v2_out, paste0("muscle_v2_markers_reversal_", ct, ".csv")), row.names = FALSE)
}
for (ct in sanitize_name(unique(markers_aged$cluster))) {
    sub_df <- markers_aged[sanitize_name(markers_aged$cluster) == ct, ]
    write.csv(sub_df, file.path(v2_out, paste0("muscle_v2_markers_aged_", ct, ".csv")), row.names = FALSE)
}

cat("\nAll v2 snRNA-seq results exported to:", v2_out, "\n")

# ── 7. CellChat with v2 reversal genes ──────────────────────────
library(CellChat)

cellchat_rgzj1 <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_muscle/cellchat_objects/cellchat_rgzj1_centr.rds")
cellchat_rgzj2 <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_muscle/cellchat_objects/cellchat_rgzj2_centr.rds")
cellchat_rgzj3 <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_muscle/cellchat_objects/cellchat_rgzj3_centr.rds")
cellchat_rgzj4 <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_muscle/cellchat_objects/cellchat_rgzj4_centr.rds")
cellchat_rgzj5 <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_muscle/cellchat_objects/cellchat_rgzj5_centr.rds")

reversal_genes_v2_clean <- na.omit(unique(rescued_to_normal$Symbol))

analyze_reversal_cellchat_v2 <- function(obj, name, rev_genes) {
    all_comm <- subsetCommunication(obj, slot.name = "net")
    lr_filt <- all_comm %>%
        filter(ligand %in% rev_genes | receptor %in% rev_genes)
    cat(name, ": LR with v2 reversal genes:", nrow(lr_filt), "\n")

    pathways_of_interest <- unique(lr_filt$pathway_name)
    cat(name, ": pathways:", paste(pathways_of_interest, collapse = ", "), "\n")

    return(lr_filt)
}

lr_ctrl_v2  <- analyze_reversal_cellchat_v2(cellchat_rgzj1, "ctrl", reversal_genes_v2_clean)
lr_age_v2   <- analyze_reversal_cellchat_v2(cellchat_rgzj2, "age", reversal_genes_v2_clean)
lr_sAct_v2  <- analyze_reversal_cellchat_v2(cellchat_rgzj3, "sAct", reversal_genes_v2_clean)
lr_DR_v2    <- analyze_reversal_cellchat_v2(cellchat_rgzj4, "DR", reversal_genes_v2_clean)
lr_combi_v2 <- analyze_reversal_cellchat_v2(cellchat_rgzj5, "combi", reversal_genes_v2_clean)

# Export CellChat LR tables
write.csv(lr_ctrl_v2,  file.path(v2_out, "ctrl_lr_interactions_reversal_genes_v2.csv"), row.names = FALSE)
write.csv(lr_age_v2,   file.path(v2_out, "age_lr_interactions_reversal_genes_v2.csv"), row.names = FALSE)
write.csv(lr_sAct_v2,  file.path(v2_out, "sAct_lr_interactions_reversal_genes_v2.csv"), row.names = FALSE)
write.csv(lr_DR_v2,    file.path(v2_out, "DR_lr_interactions_reversal_genes_v2.csv"), row.names = FALSE)
write.csv(lr_combi_v2, file.path(v2_out, "combi_lr_interactions_reversal_genes_v2.csv"), row.names = FALSE)

# ── 8. Compare CellChat v1 vs v2 ────────────────────────────────
cat("\n=== CellChat COMPARISON (original vs v2 reversal genes) ===\n")
cat(sprintf("%-8s %-15s %-15s\n", "Cond", "Original LR", "v2 LR"))
for (cond in c("ctrl", "age", "sAct", "DR", "combi")) {
    orig_path <- paste0("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/1st_attempt_count_table_data/results_tables/cellchat/",
                         cond, "_lr_interactions_reversal_genes.csv")
    if (file.exists(orig_path)) {
        orig_n <- nrow(read.csv(orig_path))
    } else {
        orig_n <- NA
    }
    v2_n <- switch(cond,
                   ctrl = nrow(lr_ctrl_v2),
                   age = nrow(lr_age_v2),
                   sAct = nrow(lr_sAct_v2),
                   DR = nrow(lr_DR_v2),
                   combi = nrow(lr_combi_v2))
    cat(sprintf("%-8s %-15s %-15d\n", cond,
                ifelse(is.na(orig_n), "not found", as.character(orig_n)), v2_n))
}

cat("\nAll CellChat v2 results exported.\n")

# ── 9. UMAP + condition exports (unchanged) ─────────────────────
# These don't depend on bulk gene lists, so no v2 needed
# But export for completeness
df_umap <- data.frame(
    celltype  = as.data.frame(py_to_r(adata$obs))$celltype,
    condition = as.data.frame(py_to_r(adata$obs))$sample,
    UMAP_1    = py_to_r(adata$obsm[["X_umap"]])[, 1],
    UMAP_2    = py_to_r(adata$obsm[["X_umap"]])[, 2]
)
write.csv(df_umap, file.path(v2_out, "muscle_v2_umap_coordinates.csv"), row.names = FALSE)

df_comp <- data.frame(
    celltype  = as.data.frame(py_to_r(adata$obs))$celltype,
    condition = as.data.frame(py_to_r(adata$obs))$sample
)
write.csv(df_comp, file.path(v2_out, "muscle_v2_celltype_condition.csv"), row.names = FALSE)

cat("UMAP and composition CSVs exported.\n")

## GOBP per celltype

In [ ]:
# ── Per-cell-type GOBP enrichment for AGED genes (v2) ───────────
# Run on server after loading v2 results

library(clusterProfiler)
library(org.Mm.eg.db)

# Load v2 aged gene markers (from FindAllMarkers overlap)
markers_aged_v2 <- read.csv(
    "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon/muscle_v2_markers_aged_genes_celltypes.csv"
)

# Load v2 deg_age for universe
deg_age_v2 <- read.csv(
    "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon/muscle_v2_deg_age.csv"
)

# All expressed genes as universe (rownames of res_age)
res_age_v2 <- read.csv(
    "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon/muscle_v2_res_age_results_with_symbols.csv"
)
all_genes_v2 <- res_age_v2$X  # Ensembl IDs

out_base <- "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon/GOBP_per_celltype/aged"
dir.create(out_base, recursive = TRUE, showWarnings = FALSE)

# Sanitize cell type names for filenames
sanitize <- function(x) gsub("[^A-Za-z0-9_]", "_", x)

cell_types <- unique(markers_aged_v2$cluster)
cat("Cell types to process:", length(cell_types), "\n")

for (ct in cell_types) {
    cat("Processing:", ct, "\n")
    
    # Get marker genes for this cell type
    ct_markers <- markers_aged_v2[markers_aged_v2$cluster == ct, "gene"]
    
    if (length(ct_markers) < 5) {
        cat("  Skipping — fewer than 5 genes\n")
        next
    }
    
    # Convert symbols to Ensembl
    ct_ensembl <- mapIds(org.Mm.eg.db, keys = ct_markers,
                          column = "ENSEMBL", keytype = "SYMBOL", multiVals = "first")
    ct_ensembl <- na.omit(ct_ensembl)
    
    if (length(ct_ensembl) < 5) {
        cat("  Skipping — fewer than 5 mappable genes\n")
        next
    }
    
    # Run enrichGO
    ego <- tryCatch(
        enrichGO(gene = ct_ensembl,
                 universe = all_genes_v2,
                 OrgDb = org.Mm.eg.db,
                 keyType = "ENSEMBL",
                 ont = "BP",
                 pAdjustMethod = "BH",
                 pvalueCutoff = 0.05,
                 qvalueCutoff = 0.2),
        error = function(e) { cat("  Error:", conditionMessage(e), "\n"); NULL }
    )
    
    if (!is.null(ego) && nrow(as.data.frame(ego)) > 0) {
        ct_safe <- sanitize(ct)
        out_path <- file.path(out_base, paste0("gobp_aged_", ct_safe, ".csv"))
        write.csv(as.data.frame(ego), out_path, row.names = FALSE)
        cat("  Saved:", nrow(as.data.frame(ego)), "terms\n")
    } else {
        cat("  No significant terms\n")
    }
}

cat("\nDone. Files in:", out_base, "\n")
list.files(out_base)

# Comparison with original enrichment

In [ ]:
# Compare top 10 aging GOBP terms
orig_aging <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/1st_attempt_count_table_data/results_tables/enrichment/muscle_enrichment_results_aging.csv")
v2_aging <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon/muscle_v2_enrichment_results_aging.csv")

cat("=== AGING GOBP: Original vs v2 ===\n")
cat("Original terms:", nrow(orig_aging), "| v2 terms:", nrow(v2_aging), "\n\n")

cat("Original top 10:\n")
print(head(orig_aging[, c("Description","Count","p.adjust")], 10))
cat("\nv2 top 10:\n")
print(head(v2_aging[, c("Description","Count","p.adjust")], 10))

# Term overlap
shared_terms <- intersect(orig_aging$Description, v2_aging$Description)
cat("\nShared terms:", length(shared_terms), "of", nrow(orig_aging), "(original) /", nrow(v2_aging), "(v2)\n")

# Repeat for DR-exclusive
orig_DR <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/1st_attempt_count_table_data/results_tables/enrichment/muscle_enrichment_results_rescued-to-normal__DR.csv")
v2_DR <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon/muscle_v2_enrichment_results_rescued-to-normal__DR.csv")

cat("\n=== DR-EXCLUSIVE GOBP: Original vs v2 ===\n")
cat("Original terms:", nrow(orig_DR), "| v2 terms:", nrow(v2_DR), "\n")
shared_DR <- intersect(orig_DR$Description, v2_DR$Description)
cat("Shared terms:", length(shared_DR), "\n")
cat("\nOriginal top 5:\n")
print(head(orig_DR[, c("Description","Count","p.adjust")], 5))
cat("\nv2 top 5:\n")
print(head(v2_DR[, c("Description","Count","p.adjust")], 5))

In [ ]:
#compare gsea
orig_gsea <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/1st_attempt_count_table_data/results_tables/muscle_fgsea_aging.csv")
v2_gsea <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon/muscle_v2_fgsea_aging.csv")

cat("=== GSEA AGING ===\n")
cat("Original sig pathways:", sum(orig_gsea$padj < 0.05 & abs(orig_gsea$NES) > 1.5, na.rm=TRUE), "\n")
cat("v2 sig pathways:", sum(v2_gsea$padj < 0.05 & abs(v2_gsea$NES) > 1.5, na.rm=TRUE), "\n")

# NES correlation for shared pathways
shared_pw <- merge(orig_gsea[, c("pathway","NES","padj")],
                    v2_gsea[, c("pathway","NES","padj")],
                    by = "pathway", suffixes = c("_orig","_v2"))
cat("Shared pathways:", nrow(shared_pw), "\n")
cat("NES correlation:", cor(shared_pw$NES_orig, shared_pw$NES_v2, use="complete.obs"), "\n")

In [ ]:
v2_gsea_sig <- v2_gsea[v2_gsea$padj < 0.05 & abs(v2_gsea$NES) > 1.5, ]
v2_gsea_sig <- v2_gsea_sig[order(v2_gsea_sig$padj), ]
print(v2_gsea_sig[, c("pathway", "NES", "padj")])

In [ ]:
#compare snRNA module scores
# Compare cell-type rankings between original and v2 module scores
orig_scores <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/1st_attempt_count_table_data/results_tables/muscle_ReversalGeneScores_perCell_seurat.csv")
v2_scores <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon/muscle_v2_module_scores_all.csv")

# Original rankings
orig_rank <- aggregate(Reversal_norm1 ~ celltype, data = orig_scores, FUN = mean)
orig_rank <- orig_rank[order(-orig_rank$Reversal_norm1), ]
cat("=== REVERSAL SCORE RANKING ===\n")
cat("Original top 5:\n")
print(head(orig_rank, 5))

v2_rank <- aggregate(Reversal_norm ~ celltype, data = v2_scores, FUN = mean)
v2_rank <- v2_rank[order(-v2_rank$Reversal_norm), ]
cat("\nv2 top 5:\n")
print(head(v2_rank, 5))

# Rank correlation
merged_ranks <- merge(orig_rank, v2_rank, by = "celltype")
cat("\nRank correlation (Spearman):", cor(merged_ranks$Reversal_norm1, merged_ranks$Reversal_norm, method = "spearman"), "\n")

In [ ]:
#compare FindAllMarkers
orig_markers <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/1st_attempt_count_table_data/results_tables/celltype_markers/muscle_markers_all_celltypes.csv")
v2_markers <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle2/results_v2_STAR_salmon/muscle_v2_markers_all_celltypes.csv")

cat("=== FindAllMarkers ===\n")
cat("Original total:", nrow(orig_markers), "| v2 total:", nrow(v2_markers), "\n")

# These should be IDENTICAL since the Seurat object didn't change
cat("Gene overlap:", length(intersect(orig_markers$gene, v2_markers$gene)),
    "of", length(unique(orig_markers$gene)), "(orig) /", length(unique(v2_markers$gene)), "(v2)\n")

In [ ]:
# Find the correct variable name for combi vs age results
cat("Available result objects:\n")
ls()[grep("res_|intv|combi", ls())]

# Check res_intv (without _df suffix)
cat("\n=== Combi vs age LFCs (res_intv) ===\n")
res_intv_check <- as.data.frame(res_intv)
res_intv_check$Symbol <- mapIds(org.Mm.eg.db,
    keys = sub("\\.[0-9]+$", "", rownames(res_intv_check)),
    column = "SYMBOL", keytype = "ENSEMBL", multiVals = "first")
intv_hits <- res_intv_check[!is.na(res_intv_check$Symbol) &
                             res_intv_check$Symbol %in% genes_of_interest,
                             c("Symbol","log2FoldChange","padj")]
print(intv_hits)

# Combi vs ctrl
cat("\n=== Combi vs ctrl LFCs (res_combi_ctrl) ===\n")
res_cc_check <- as.data.frame(res_combi_ctrl)
res_cc_check$Symbol <- mapIds(org.Mm.eg.db,
    keys = sub("\\.[0-9]+$", "", rownames(res_cc_check)),
    column = "SYMBOL", keytype = "ENSEMBL", multiVals = "first")
cc_hits <- res_cc_check[!is.na(res_cc_check$Symbol) &
                         res_cc_check$Symbol %in% genes_of_interest,
                         c("Symbol","log2FoldChange","padj")]
print(cc_hits)

---
# Pipeline metadata

| Parameter | Value |
|---|---|
| Reference genome | GRCm39 / Ensembl 110 |
| Trimming | TrimGalore (via nf-core v3.16 cached) |
| Alignment | STAR v2.7.10a (two-pass, manual pipeline) |
| Quantification | Salmon (from STAR transcriptome BAM) |
| Import | tximport (tx2gene built from GTF) |
| DGE | DESeq2 + lfcShrink (ashr) |
| Expression filter | \u226510 total counts |
| Mapping rates | 88.3\u201390.9% |
| Matches kidney pipeline | \u2705 Yes (same genome, aligner, quantifier, contrasts) |
| Replaces | FeatureCounts/GRCm38 pre-processed counts from GEO |